# Run in colab

In [1]:
!pip install instructor

In [2]:
!cd /content
!rm -rf macro_financial_forecasting

In [3]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 1692, done.
remote: Counting objects: 100% (681/681), done.
remote: Compressing objects: 100% (248/248), done.
remote: Total 1692 (delta 510), reused 458 (delta 433), pack-reused 1011 (from 1)
Receiving objects: 100% (1692/1692), 19.11 MiB | 19.99 MiB/s, done.
Resolving deltas: 100% (1041/1041), done.


In [4]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


In [5]:
from config import Config
from train_data_loader import TrainDataLoader
from data_model.bloomberg_news_entry import BloombergNewsEntry
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")

train_data_loader = TrainDataLoader(config)

In [6]:
print("Starting loading pipeline ...")
print(f"Config: {config}")

train_ds = train_data_loader.load()
print("Loading pipeline completed.")

Starting loading pipeline ...
Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'http

bloomberg_financial_data.parquet.gzip:   0%|          | 0.00/482M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/446762 [00:00<?, ? examples/s]


--- Download Successful! ---


Map:   0%|          | 0/446762 [00:00<?, ? examples/s]

Training dataset processed.

--- Starting validation of 446762 entries ---


Validating entries: 100%|██████████| 446762/446762 [00:27<00:00, 16422.90it/s]


--- Validation Complete! ---
Training dataset validated.
Saving processed dataset to local cache at '../data/danidanou_Bloomberg_Financial_News_train'...
Total number of rows: 446762
Loading pipeline completed.


In [7]:
train_ds[0]

BloombergNewsEntry(Headline='Ivory Coast Keeps Cocoa Export Tax Below 22%, Document Shows', Date='2011-10-06', Link='http://www.bloomberg.com/news/2011-10-06/ivory-coast-keeps-cocoa-export-tax-below-22-document-shows.html', Article='Export taxes on cocoa beans from Ivory Coast , the world’s biggest producer of the chocolate ingredient, won’t exceed 22 percent of the international price this season, meeting a commitment to the International Monetary Fund , according to a finance ministry document. In the 2008-9 season taxes averaged 25.3 percent of international prices, the IMF said in a document posted on its website in November last year. While the country met the commitment in the season just ended, it had a change in government earlier this year. The rate meets a demand by the International Monetary Fund and the World Bank to reform the Ivorian cocoa and coffee industries in order to comply with the terms of its Heavily Indebted Poor Countries’ debt-relief program. Last year, the fi

## Update these 2 variables to run pipeline on

In [8]:
DATA_START=0
DATA_END=20

In [9]:
from processor import NewsProcessor
import nest_asyncio
nest_asyncio.apply()

processor = NewsProcessor(config)
results = await processor.transduce_news_entries_async(train_ds[DATA_START:DATA_END], save_path_prefix="processed_news") #Sample 50 news

Processing news:   0%|          | 0/20 [00:00<?, ?entry/s]


AttributeError: 'ProxyClientChat' object has no attribute 'acall'

In [ ]:
processed_df = processor.group_by_date_and_industry(results, save_path="grouped_data")

In [35]:
processed_df

,Industry,Date,News
0,Communication Services,2007-11-02,"[{'Article': 'OAO Comstar United Telesystems, ..."
1,Communication Services,2010-02-05,"[{'Article': 'Tony Hall, the 58-year-old chief..."
2,Communication Services,2010-06-14,"[{'Article': 'Le Monde, France’s newspaper of ..."
3,Communication Services,2011-08-03,"[{'Article': 'Telecom Italia SpA (TIT) , the n..."
4,Communication Services,2011-08-04,"[{'Article': 'Netia SA (NET) , Poland’s second..."
...,...,...,...
667,Utilities,2013-04-30,"[{'Article': 'Iberdrola SA (IBE) , the Spanish..."
668,Utilities,2013-07-10,"[{'Article': 'Cia. Paranaense de Energia , Bra..."
669,Utilities,2013-07-17,[{'Article': 'The U.K.’s longest heat wave in ...
670,Utilities,2013-07-19,"[{'Article': 'Australia ’s New South Wales, th..."


In [36]:
processed_df = await processor.process_dataframe(processed_df, save_path="sentiment_data")

Explanation: 100%|██████████| 672/672 [02:55<00:00,  3.83it/s]


In [38]:
processed_df

,Industry,Date,News,Summary,SentimentScore,SentimentExplanation
0,General Market,2007-09-07,"[{'Article': 'Russian stocks dropped, led by O...","- U.S. payrolls fell 4,000 in August, first de...",0.011075,The near-zero FinBERT score indicates a neutra...
1,Consumer Discretionary,2007-09-07,"[{'Article': 'Etam Developpement SA (CS) , Fra...","Etam Developpement SA, France's largest public...",0.010423,Near-neutral sentiment (FinBERT ≈ 0.01) driven...
2,Communication Services,2007-11-02,"[{'Article': 'OAO Comstar United Telesystems, ...",- Comstar United Telesystems reported a third-...,0.012233,The FinBERT score of 0.012 indicates a near-ne...
3,Real Estate,2007-12-13,"[{'Article': 'Centro Properties Group , Austra...",Centro Properties Group plans to revise its ea...,0.015082,Explanation: The FinBERT score of 0.015 sugges...
4,Financials,2007-12-13,[{'Article': 'Irish Life & Permanent Plc fell ...,- Irish Life & Permanent warned 2008 operating...,0.010902,FinBERT score of 0.011 indicates near-neutral ...
...,...,...,...,...,...,...
667,Industrials,2013-07-28,[{'Article': '(Corrects Nissan’s reporting dat...,"- GM outsold Toyota in the June quarter, deliv...",-0.005119,FinBERT score -0.005 suggests near-neutral sen...
668,Real Estate,2013-07-29,[{'Article': '(Corrects size of loss in first ...,"- Homex posts a 10.2 billion peso loss, its bi...",0.011860,The FinBERT score of 0.012 implies near-neutra...
669,Industrials,2013-07-29,[{'Article': '(Corrects spelling of hometown i...,Summary\n- Dennis Dammerman died July 23 at 67...,0.523384,The FinBERT score of 0.523 implies a mildly po...
670,Health Care,2013-07-29,"[{'Article': 'Barnaby Jack, the 36-year-old ha...",Barnaby Jack was a computer-security researche...,0.815967,The FinBERT score of 0.816 indicates a positiv...


In [40]:
pd.read_parquet("../data/sentiment_data")

,Industry,Date,News,Summary,SentimentScore,SentimentExplanation
0,General Market,2007-09-07,"[{'Article': 'Russian stocks dropped, led by O...","- U.S. payrolls fell 4,000 in August, first de...",0.011075,The near-zero FinBERT score indicates a neutra...
1,Consumer Discretionary,2007-09-07,"[{'Article': 'Etam Developpement SA (CS) , Fra...","Etam Developpement SA, France's largest public...",0.010423,Near-neutral sentiment (FinBERT ≈ 0.01) driven...
2,Communication Services,2007-11-02,"[{'Article': 'OAO Comstar United Telesystems, ...",- Comstar United Telesystems reported a third-...,0.012233,The FinBERT score of 0.012 indicates a near-ne...
3,Real Estate,2007-12-13,"[{'Article': 'Centro Properties Group , Austra...",Centro Properties Group plans to revise its ea...,0.015082,Explanation: The FinBERT score of 0.015 sugges...
4,Financials,2007-12-13,[{'Article': 'Irish Life & Permanent Plc fell ...,- Irish Life & Permanent warned 2008 operating...,0.010902,FinBERT score of 0.011 indicates near-neutral ...
...,...,...,...,...,...,...
667,Industrials,2013-07-28,[{'Article': '(Corrects Nissan’s reporting dat...,"- GM outsold Toyota in the June quarter, deliv...",-0.005119,FinBERT score -0.005 suggests near-neutral sen...
668,Real Estate,2013-07-29,[{'Article': '(Corrects size of loss in first ...,"- Homex posts a 10.2 billion peso loss, its bi...",0.011860,The FinBERT score of 0.012 implies near-neutra...
669,Industrials,2013-07-29,[{'Article': '(Corrects spelling of hometown i...,Summary\n- Dennis Dammerman died July 23 at 67...,0.523384,The FinBERT score of 0.523 implies a mildly po...
670,Health Care,2013-07-29,"[{'Article': 'Barnaby Jack, the 36-year-old ha...",Barnaby Jack was a computer-security researche...,0.815967,The FinBERT score of 0.816 indicates a positiv...


## JIT Fix

In [39]:
import pandas as pd
from utils.pydantic_parquet_util import ParquetUtil

results = pd.read_parquet("../data/processed_news_batch_1")

In [22]:
df = (
    pd.read_parquet("../data/processed_news_batch_1").groupby(['Industry', 'Date'])
    .apply(lambda x: x.to_dict(orient='records'))
    .reset_index()
    .rename(columns={0: 'News'})
)
ParquetUtil.save_df_to_parquet(df, "../data/grouped_data")

/tmp/ipython-input-1032707156.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.to_dict(orient='records'))
